<a href="https://colab.research.google.com/github/lsmc-isa/ML_project-/blob/main/Final_Project_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plant Disease Classification using Transfer Learning

**Master's in Green Data Science 2025-2026**<br>
**Practical Machine Learning**<br>
**Team Members:** João Lopes (24842) & Lourenço Carvalho (29248)

---

## Table of Contents
1. [Introduction](#1.-Introduction)
2. [Data](#2.-Data)
3. [Data Organization](#3.-Data-Organization)
4. [Methods](#4.-Methods)
5. [Results](#5.-Results)
6. [Analysis](#6.-Analysis)
7. [Deployment](#7.-Deployment)
8. [References](#8.-References)
9. [Contributions](#9.-Contributions)

<h2 id="1.-Introduction">1. Introduction</h2>

Plant diseases have a devastating impact on agricultural yield, causing significant economic losses and threatening global food security. Early, accurate, and rapid detection of these diseases is crucial to mitigate their spread. Traditionally, disease identification requires manual inspection by domain experts, which is time-consuming, expensive, and prone to human error.

This project investigates how computer vision can automate this process. It bridges the gap between advanced deep learning techniques and real-world agricultural challenges, empowering farmers with accessible and reliable diagnostic tools.

<h2 id="2.-Data">2. Data</h2>

We utilized the **PlantVillage dataset**, a well-established and publicly available dataset widely used in agricultural computer vision research. It contains tens of thousands of high-resolution images of healthy and diseased crop leaves across various species (e.g., apple, corn, potato, tomato). The dataset was collected from Kaggle using the `opendatasets` library.

### Data Cleaning & Feature Engineering
Since the dataset consists of image data, traditional tabular cleaning was not applicable. Instead, we performed image preprocessing and data augmentation to handle the challenges of high intra-class variation and prevent overfitting:
- **Resizing:** All images were resized to 224x224 pixels to match the input requirements of our pre-trained networks.
- **Data Augmentation:** For the training set, we applied Random Rotation (40 degrees), Random Resized Crop (scale 0.8 to 1.0), and Random Horizontal Flips.
- **Normalization:** Images were converted to PyTorch Tensors and normalized using ImageNet standard mean `[0.485, 0.456, 0.406]` and standard deviation `[0.229, 0.224, 0.225]`.

In [ ]:
# Note on AI Generation:
# prompt: Please provide the code to download the dataset from Kaggle and set up the PyTorch DataLoaders with the necessary transformations.

!pip install opendatasets
import opendatasets as od
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# Download Dataset
dataset_url = 'https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset'
od.download(dataset_url)

# Set up paths
dataset_path = './new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
train_dir = os.path.join(dataset_path, 'train')
valid_dir = os.path.join(dataset_path, 'valid')

img_height, img_width = 224, 224
batch_size = 32

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.RandomRotation(40),
    transforms.RandomResizedCrop(img_height, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create Datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(valid_dir, transform=val_transforms)

num_classes_pytorch = len(train_dataset.classes)
print(f"Detected {num_classes_pytorch} classes.")

<h2 id="3.-Data-Organization">3. Data Organization</h2>

The downloaded dataset was already organized into standard folders for image classification.
- **Training Set:** Comprises approximately 70,295 images used to fit the model parameters.
- **Validation Set:** Comprises 17,572 images used to evaluate the model during training and apply early stopping to prevent overfitting.
- **DataLoaders:** We used PyTorch's `DataLoader` with a batch size of 32 and shuffling enabled for the training set.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

<h2 id="4.-Methods">4. Methods</h2>

Training a deep neural network from scratch requires massive amounts of data and computational power. We leveraged **Transfer Learning** using pre-trained Convolutional Neural Networks (CNNs).

We compared two distinct architectures available in PyTorch:
1. **ResNet50:** A deep, highly capable architecture (approx. 15 million trainable parameters) known for its residual connections that solve the vanishing gradient problem. It acts as our accuracy benchmark.
2. **MobileNetV2:** A lightweight architecture (approx. 48 thousand trainable parameters in the classifier) optimized for mobile and edge devices, utilizing depthwise separable convolutions.

**Architecture Choices:** For both models, we froze the base convolutional layers to retain their ImageNet-learned feature extraction capabilities (edges, textures). We then replaced the final fully connected layers (`fc` in ResNet, `classifier` in MobileNet) to output predictions for our specific 38 plant disease classes. We used the **Adam optimizer** with a learning rate of `1e-4` and **CrossEntropyLoss**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Setup ResNet50 ---
resnet_model_pt = torchvision.models.resnet50(weights='DEFAULT')
for param in resnet_model_pt.parameters():
    param.requires_grad = False
for param in resnet_model_pt.layer4.parameters(): # Unfreeze last block
    param.requires_grad = True
resnet_model_pt.fc = nn.Linear(resnet_model_pt.fc.in_features, num_classes_pytorch)
resnet_model_pt = resnet_model_pt.to(device)

# --- Setup MobileNetV2 ---
mobilenet_model_pt = torchvision.models.mobilenet_v2(weights='DEFAULT')
for param in mobilenet_model_pt.parameters():
    param.requires_grad = False
for param in mobilenet_model_pt.classifier.parameters(): # Unfreeze classifier
    param.requires_grad = True
in_features = mobilenet_model_pt.classifier[-1].in_features
mobilenet_model_pt.classifier[-1] = nn.Linear(in_features, num_classes_pytorch)
mobilenet_model_pt = mobilenet_model_pt.to(device)

# (The train_model_pytorch function definition goes here. Omitted for brevity in this cell but assume defined as in original code)

<h2 id="5.-Results">5. Results</h2>

We ran both models for 4 epochs to establish a baseline performance. The results are summarized below:

| Model | Train Loss | Train Accuracy | Val Loss | Val Accuracy |
| :--- | :---: | :---: | :---: | :---: |
| **ResNet50** | ~0.016 | ~99.4% | ~0.015 | ~99.5% |
| **MobileNetV2** | ~0.419 | ~90.1% | ~0.383 | ~90.8% |

<h2 id="6.-Analysis">6. Analysis</h2>

The results demonstrate the fundamental trade-off in deep learning architectures.

**ResNet50** achieved exceptional accuracy (>99%) very quickly. Its depth and residual connections allow it to extract highly complex patterns, easily distinguishing between diseases with low inter-class variation. However, this comes at the cost of high computational load and slower inference times.

**MobileNetV2** achieved a respectable ~90.8% accuracy. While lower than ResNet50, this is expected given its lightweight nature. Crucially, as we identified during the project, the performance of MobileNetV2 can be significantly improved with further training adjustments:
1. **More Epochs:** 4 epochs is insufficient for MobileNetV2 to fully converge.
2. **Deep Fine-Tuning:** Unfreezing more convolutional blocks (rather than just the classifier) would allow the model to adapt its feature extraction specifically to plant textures.
3. **Learning Rate Scheduler:** Implementing a learning rate decay would help the lightweight model settle into a more optimal minimum without overshooting.

Ultimately, while ResNet50 is the better model for a server-side solution, MobileNetV2 remains the most viable option for our real-world goal: deploying directly onto a farmer's smartphone in the field without requiring an internet connection.

<h2 id="7.-Deployment">7. Deployment (Gradio on Hugging Face)</h2>

To make our models accessible to the final user, we developed a web interface using **Gradio**. This interface allows a user to upload a picture of a leaf, select the backend model (ResNet50 or MobileNetV2), and instantly receive a prediction with confidence scores. The application code is prepared for deployment on **Hugging Face Spaces**.

In [ ]:
!pip install gradio
import gradio as gr

# Note: In the final app.py, models are loaded via torch.load('best_ResNet_pytorch.pth')
# The predict_image_pytorch function handles inference

# Example of launching the interface:
'''
demo = gr.Interface(
    fn=predict_image_pytorch,
    inputs=[
        gr.Image(type="numpy", label="Upload de Imagem de Folha de Planta"),
        gr.Radio(["ResNet50", "MobileNetV2"], label="Escolha do Modelo")
    ],
    outputs=[
        gr.Label(label="Resultado da Previsão"),
    ],
    title="Classificação de Doenças de Plantas",
)
demo.launch()
'''

<h2 id="8.-References">8. References</h2>

- Hughes, D., & Salathe, M. (2015). An open access repository of images on plant health to enable the development of mobile disease diagnostics (PlantVillage).
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep Residual Learning for Image Recognition (ResNet). CVPR.
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., & Chen, L. (2018). MobileNetV2: Inverted Residuals and Linear Bottlenecks. CVPR.
- PyTorch Documentation: https://pytorch.org/docs/stable/index.html
- Gradio Documentation: https://gradio.app/docs/

<h2 id="9.-Contributions">9. Contributions</h2>

- **João Lopes (24842):
- **Lourenço Carvalho (29248):